# NYC EV Fleet Station Network

| | |
|---|---|
| **Source** | Department of Citywide Administrative Services (DCAS) |
| **URL** | [data.cityofnewyork.us](https://data.cityofnewyork.us/City-Government/NYC-EV-Fleet-Station-Network/fc53-9hrv) |
| **Granularity** | One row per charging station |
| **Data collection** | DCAS-operated stations via ChargePoint; non-DCAS data from the respective operating agencies |

DCAS operates over 1,025 EV charging stations with 1,600+ ports across city garages and parking facilities, including 180+ fast-charging stations and 80+ solar carports. A subset of stations is open to the public for a nominal fee. NYC DOT separately operates a curbside public charging network.

## Column Reference

| Column | Description | Notes |
|---|---|---|
| `agency` | Abbreviation of the city agency where the station is located | See [Agency Codes](#agency-codes) below |
| `station_name` | Name of the EV charging station | |
| `type_of_charger` | Charger type at the station | See [Charger types](#charger-types) below |
| `no_of_ports` | Number of charging ports at the station | |
| `public_charger_` | Whether the station accepts public users | See [Public charging](#public-charging) |
| `fee_for_city_drivers` | Whether city fleet drivers pay a fee | Blank = no fee |
| `street` | Street address of the station | |
| `city` | City of the station's location | |
| `postcode` | ZIP code of the station's location | |
| `borough` | NYC borough of the station's location | |

### Geocoded Columns

The following columns were generated by the NYC Open Data team (Office of Technology and Innovation) through geocoding and are not self-reported by DCAS.

| Column | Description |
|---|---|
| `latitude` / `longitude` | WGS84 coordinates of the station address |
| `community_district` | One of 59 community districts used for municipal service delivery |
| `council_district` | One of 51 NYC City Council districts |
| `bin` | 7-digit Building Identification Number assigned by DCP; first digit encodes the borough |
| `bbl` | Borough–Block–Lot identifier assigned by DOF, used for tax and land use purposes |
| `census_tract` | 2020 census tract; note that leading/trailing zeros are dropped |
| `nta` | Neighborhood Tabulation Area (NTA) (2020); DCP-defined small area (~15,000 min. population); boundaries do not definitively represent neighborhoods |

### Agency Codes

| Code | Agency |
|---|---|
| ACS | Administration for Children's Services |
| CITYHALL | City Hall |
| DCAS | Department of Citywide Administrative Services |
| DDC | Department of Design & Construction |
| DEP | Department of Environmental Protection |
| DHS | Department of Homeless Services |
| DOC | Department of Correction |
| DOE | Department of Education |
| DOH | Department of Health & Mental Hygiene |
| DOT | Department of Transportation |
| DPR | Department of Parks & Recreation |
| DSNY | Department of Sanitation |
| FDNY | Fire Department of New York |
| HPD | Housing Preservation & Development |
| HRA | Human Resources Administration |
| NYCEM | Office of Emergency Management |
| NYCHA | New York City Housing Authority |
| NYPD | New York Police Department |
| OCME | Office of Chief Medical Examiner |
| PROB | Department of Probation |
| TLC | Taxi & Limousine Commission |

### Charger types

DOT chargers are DOT public chargers available for City fleet use. DOT Flo chargers are curbside public chargers available for City fleet use. The following values indicate the type of charger at the station:

- `DOT Municipal Level 2`
- `DOT Municipal Level 3`
- `EV Solar Arc`
- `EV Solar Canopy`
- `L2 DOT Flo Curbside`
- `Level 2`
- `Level 3 Fast Charger`
- `Mobile Charger`

### Public charging

Blank values indicate that the station is not publicly available. City Hall Charging is only available for employees working at City Hall. The following values indicate that the station is open to public users:

- `Yes`
- `No`
- `City Hall Charging`
- `DOT Municipal Parking`
- `NYC DC Fast Public Charging`
- `Solar Carport Charging`
- `*adapter required`


## Data Acquisition

In [1]:
import json
import requests
import folium
from datetime import date
from pathlib import Path
from folium.plugins import MarkerCluster
import pandas as pd
import geopandas as gpd

In [2]:
ROOT = Path("..").resolve()
URL = "https://data.cityofnewyork.us/resource/fc53-9hrv"
PARAMS = {"$limit": 10000}
CSV_PATH = ROOT / "data/raw/dcas_ev_stations.csv"
META_PATH = ROOT / "data/raw/dcas_ev_stations.csv.meta.json"

if CSV_PATH.exists():
    print(f"Already fetched → {CSV_PATH}, skipping.")
else:
    CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    print(f"Fetching dataset → {CSV_PATH}")
    resp = requests.get(f"{URL}.csv", params=PARAMS, timeout=30)
    resp.raise_for_status()
    CSV_PATH.write_bytes(resp.content)
    META_PATH.write_text(
        json.dumps(
            {
                "fetched_date": str(date.today()),
                "source_url": URL,
                "source_params": PARAMS,
                "row_count": len(resp.text.splitlines()) - 1,
            },
            indent=2,
        )
    )
    print(f"Saved {CSV_PATH.name} ({META_PATH.read_text()})")

pd.read_csv(CSV_PATH)

Already fetched → /home/henrique/agentwin-dataset/data/raw/dcas_ev_stations.csv, skipping.


,agency,station_name,type_of_charger,no_of_ports,street,city,postcode,borough,public_charger_,fee_for_city_drivers,latitude,longitude,community_district,council_district,census_tract,bin,bbl,nta
0,ACS,NYC FLEET / ACS_LINDEN_1_L3,Level 3 Fast Charger,1.0,2554 Linden Blvd,East New York,11208.0,Brooklyn,NaN,NaN,40.668034,-73.869949,305.0,42.0,1220.0,3098733.0,3.044840e+09,BK0505
1,ACS,NYC FLEET / ACS_LINDEN_2_L3,Level 3 Fast Charger,1.0,2554 Linden Blvd,East New York,11208.0,Brooklyn,NaN,NaN,40.668034,-73.869949,305.0,42.0,1220.0,3098733.0,3.044840e+09,BK0505
2,ACS,NYC FLEET / ACS_LINDEN-1,Level 2 Charger,1.0,2554 Linden Blvd,Brooklyn,11208.0,Brooklyn,NaN,NaN,40.668034,-73.869949,305.0,42.0,1220.0,3098733.0,3.044840e+09,BK0505
3,ACS,NYC FLEET / ACSSTMARKSPL1,Level 2 Charger,1.0,350 St Marks Pl,Staten Island,10301.0,Staten Island,NaN,NaN,40.640708,-74.077697,501.0,49.0,3.0,5000185.0,5.000160e+09,SI0101
4,CITYHALL,NYC FLEET / CITYHALL EAST,Level 2 Charger,2.0,New York City Hall,New York,10007.0,Manhattan,NaN,NaN,40.712806,-74.006096,101.0,1.0,31.0,1079147.0,1.001220e+09,MN0102
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1680,DCAS,DCAS_BXHOJ-FORDPRO-1,Level 2 Charger,1.0,265 E 161st St,Bronx,10451.0,Bronx,NaN,NaN,40.825928,-73.919514,204.0,16.0,18302.0,2097114.0,2.024440e+09,BX0401
1681,DEP,NYC FLEET / DEP_SHAFT18-5,Level 2 Charger,2.0,20 Westlake Dr,Valhalla,10595.0,Westchester,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1682,DOT,NYC FLEET / DOT_FLTBSH_2_L3,Level 3 Fast Charger,1.0,2900 Flatbush Avenue,Brooklyn,11234.0,Brooklyn,NaN,NaN,40.600248,-73.912051,318.0,46.0,666.0,3397327.0,3.085901e+09,BK1891
1683,DEP,NYC FLEET / DEP_SHAFT18-6,Level 2 Charger,2.0,20 Westlake Dr,Valhalla,10595.0,Westchester,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Preprocessing

The raw CSV requires six cleaning steps before it can be used in the evaluation scenario.

1. **Drop stations without coordinates** - Stations missing `latitude` or `longitude` cannot participate in the geographic–topological bridge, which requires a valid WGS84 position to correlate each station with a ConEdison substation zone. The data dictionary notes that coordinates are geocoded by the Open Data team and may be absent for a small number of records.
2. **Fix column types** - `no_of_ports` is ingested as float because pandas promotes integer columns containing `NaN` to float; after step 1 the column is clean and can be safely cast. `postcode` must be stored as a zero-padded string to preserve codes with leading zeros (e.g. `07030`).
3. **Fill semantically empty fields** - The data dictionary explicitly states that blank values in `public_charger_` and `fee_for_city_drivers` mean the condition does not apply - equivalent to `"No"`. Retaining `NaN` would misrepresent these fields and produce incorrect results when filtering stations by public availability.
4. **Aggregate to station granularity** - The [data dictionary](https://data.cityofnewyork.us/api/views/fc53-9hrv/files/635c379a-691f-4986-8f5c-300b15cec89d?download=true&filename=NYC%20EV%20Fleet%20Station%20Network_Data%20Dictionary.xlsx) describes each record as a charging station. However, inspection of the raw export shows multiple records sharing the same agency and geocoded coordinates, often with different charger types. We therefore treat these records as charger-type entries and aggregate them into one station per (agency, latitude, longitude), summing no_of_ports and retaining the distinct charger types.
5. **Build GeoDataFrame** - Station coordinates are projected to EPSG:4326 (WGS84), the implicit CRS of GeoJSON per RFC 7946.

In [3]:
df = pd.read_csv(CSV_PATH)
print(f"Fetched {len(df)} rows from {CSV_PATH.name}")

# 1. Drop stations without coordinates
before = len(df)
df = df.dropna(subset=["latitude", "longitude"])
print(f"Dropped {before - len(df)} stations without coordinates - {len(df)} retained")

# 2. Fix column types
df["no_of_ports"] = df["no_of_ports"].astype(int)
df["postcode"] = df["postcode"].astype(int).astype(str).str.zfill(5)

# 3. Fill semantically empty fields (blank = "No" per data dictionary)
df["public_charger_"] = df["public_charger_"].fillna("No")
df["fee_for_city_drivers"] = df["fee_for_city_drivers"].fillna("No")

# 4. Aggregate to station granularity
before = len(df)
df = df.groupby(["agency", "latitude", "longitude"], sort=False).agg(
    agency=("agency", "first"),
    station_name=("street", lambda s: f"{df.loc[s.index, 'agency'].iloc[0]} - {s.iloc[0]}, {df.loc[s.index, 'borough'].iloc[0]}"),
    charger_types=("type_of_charger", lambda s: sorted(s.dropna().unique().tolist())),
    no_of_ports=("no_of_ports", "sum"),
    street=("street", "first"),
    city=("city", "first"),
    postcode=("postcode", "first"),
    borough=("borough", "first"),
    public_charger_=("public_charger_", "first"),
    fee_for_city_drivers=("fee_for_city_drivers", "first"),
    latitude=("latitude", "first"),
    longitude=("longitude", "first"),
    community_district=("community_district", "first"),
    council_district=("council_district", "first"),
    census_tract=("census_tract", "first"),
    bin=("bin", "first"),
    bbl=("bbl", "first"),
    nta=("nta", "first"),
).reset_index(drop=True)
print(f"Grouped {before - len(df)} stations by agency, latitude and longitude - {len(df)} retained")

# 5. Build GeoDataFrame
ev_stations = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326",
)

ev_stations.head()

Fetched 1685 rows from dcas_ev_stations.csv


Dropped 318 stations without coordinates - 1367 retained
Grouped 849 stations by agency, latitude and longitude - 518 retained


,agency,station_name,charger_types,no_of_ports,street,city,postcode,borough,public_charger_,fee_for_city_drivers,latitude,longitude,community_district,council_district,census_tract,bin,bbl,nta,geometry
0,ACS,"ACS - 2554 Linden Blvd, Brooklyn","[Level 2 Charger, Level 3 Fast Charger]",3,2554 Linden Blvd,East New York,11208,Brooklyn,No,No,40.668034,-73.869949,305.0,42.0,1220.0,3098733.0,3.044840e+09,BK0505,POINT (-73.86995 40.66803)
1,ACS,"ACS - 350 St Marks Pl, Staten Island",[Level 2 Charger],1,350 St Marks Pl,Staten Island,10301,Staten Island,No,No,40.640708,-74.077697,501.0,49.0,3.0,5000185.0,5.000160e+09,SI0101,POINT (-74.0777 40.64071)
2,CITYHALL,"CITYHALL - New York City Hall, Manhattan",[Level 2 Charger],4,New York City Hall,New York,10007,Manhattan,No,No,40.712806,-74.006096,101.0,1.0,31.0,1079147.0,1.001220e+09,MN0102,POINT (-74.0061 40.71281)
3,DCAS,"DCAS - 1 Centre St, Manhattan","[EV Solar Arc Charger, Level 3 Fast Charger]",3,1 Centre St,New York,10007,Manhattan,No,No,40.713001,-74.004181,101.0,1.0,2901.0,1001394.0,1.001210e+09,MN0102,POINT (-74.00418 40.713)
4,DCAS,"DCAS - 2 Navy St, Brooklyn","[Level 2 Charger, Level 3 Fast Charger]",5,2 Navy St,Brooklyn,11201,Brooklyn,No,No,40.700610,-73.980420,302.0,35.0,23.0,NaN,NaN,BK0202,POINT (-73.98042 40.70061)


In [ ]:
map = folium.Map(location=[ev_stations.geometry.y.mean(), ev_stations.geometry.x.mean()],
               zoom_start=11)

# cluster = MarkerCluster().add_to(map)
cluster = map

for _, station in ev_stations.iterrows():
    folium.Marker(
        location=[station.geometry.y, station.geometry.x],
        tooltip=station['station_name'],
        popup=folium.Popup(f"""
                <b>{station['station_name']}</b><br>
                Borough: {station['borough']}<br>
                Ports: {station['no_of_ports']}<br>
                Charger Types: {', '.join(station['charger_types'])}<br>
                """, max_width=300),
        icon=folium.Icon(prefix="fa", icon="charging-station", color="darkblue", icon_color="white"),
        # icon=folium.Icon(prefix="fa", icon="industry", color="darkred", icon_color="white"),
        
        
    ).add_to(cluster)

map

6. **Derive ontology fields** - Three fields required by the `EVChargingStation` DTDL schema cannot be read directly from the source data and are derived here. `socket_type` maps the aggregated `charger_types` list to the single most capable `socketType` enum value: any Level 3 or DC fast charger maps to `CCSSAE` (CCS/SAE, the dominant US DC fast-charging standard); otherwise Level 2 AC chargers map to `J1772` (SAE J-1772, the universal US Level 2 standard); unrecognised types (e.g. Mobile Charger) map to `Other`. Because DTDL v2 enum properties are single-valued, "most capable" is the appropriate representative value for a facility with mixed charger types. `status` and `available_capacity` are simulation-controlled fields with no counterpart in the static source data; they are initialised to `working` and `no_of_ports` respectively, representing a nominal operating state in which all ports are available. These defaults are overridden per-scenario in the Simulation Scenario section below.

In [ ]:
# # 6. Derive ontology fields

# # socketType: most capable connector standard present at the facility
# _LEVEL3_KEYWORDS = ("Level 3", "Fast", "Municipal Level 3")
# _LEVEL2_KEYWORDS = ("Level 2", "Solar", "Flo", "Municipal Level 2")

# def _most_capable_socket_type(charger_types: list) -> str:
#     if any(kw in t for t in charger_types for kw in _LEVEL3_KEYWORDS):
#         return "CCSSAE"
#     if any(kw in t for t in charger_types for kw in _LEVEL2_KEYWORDS):
#         return "J1772"
#     return "Other"

# ev_stations_dtwin = ev_stations_agg.copy()
# ev_stations_dtwin["socket_type"] = ev_stations_dtwin["charger_types"].apply(_most_capable_socket_type)

# # status and availableCapacity: simulation defaults - overridden per scenario below
# ev_stations_dtwin["status"] = "working"
# ev_stations_dtwin["available_capacity"] = ev_stations_dtwin["no_of_ports"]

# print(ev_stations_dtwin["socket_type"].value_counts().to_string())
# ev_stations_dtwin[["station_name", "charger_types", "socket_type", "status", "available_capacity"]].head(10)

socket_type
J1772     377
CCSSAE    141


,station_name,charger_types,socket_type,status,available_capacity
0,"ACS - 2554 Linden Blvd, Brooklyn","[Level 2 Charger, Level 3 Fast Charger]",CCSSAE,working,3
1,"ACS - 350 St Marks Pl, Staten Island",[Level 2 Charger],J1772,working,1
2,"CITYHALL - New York City Hall, Manhattan",[Level 2 Charger],J1772,working,4
3,"DCAS - 1 Centre St, Manhattan","[EV Solar Arc Charger, Level 3 Fast Charger]",CCSSAE,working,3
4,"DCAS - 2 Navy St, Brooklyn","[Level 2 Charger, Level 3 Fast Charger]",CCSSAE,working,5
5,"DCAS - 6626 Metropolitan Avenue, Queens","[Level 2 Charger, Level 3 Fast Charger]",CCSSAE,working,8
6,"DCAS - 390 Kent Ave, Brooklyn","[Level 2 Charger, Level 3 Fast Charger]",CCSSAE,working,7
7,"DCAS - 350 Marconi Street, Bronx",[Level 3 Fast Charger],CCSSAE,working,4
8,"DCAS - 120-55 Queens Blvd, Queens",[Level 3 Fast Charger],CCSSAE,working,1
9,"DCAS - 100 Gold St, Manhattan","[Level 2 Charger, Level 3 Fast Charger]",CCSSAE,working,6


# TODO

- [x] Move GeoDataFrame creation to the end of the preprocessing pipeline
- [x] Fix aggregation to preserve original columns and order
- [ ] Embed step 6 as part of step 5 aggregation (type_of_charger)
    - [ ] Consider to use **type3** instead of **J1772** and **type2** instead of **CCSSAE**.
- [ ] Set the ontology step 6 as comment to be promoted to another section or notebook in the near future. 
    - [ ] The ontology fields are not part of the source data and are derived for the simulation scenario, so they may be better placed in a separate notebook that is specific to the simulation scenario.
